In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    from_json, col, window, count, sum as spark_sum,
    current_timestamp, lit, to_json, struct
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    TimestampType, BooleanType
)

spark = SparkSession.builder \
    .appName('WaveGuard_FraudDetector') \
    .config('spark.jars.packages',
            'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0') \
    .config('spark.sql.streaming.checkpointLocation', '/tmp/waveguard_checkpoint') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print("Spark Session créée :", spark.version)

Spark Session créée : 3.5.0


In [3]:
schema = StructType([
    StructField('transaction_id', StringType(), True),
    StructField('timestamp', TimestampType(), True),
    StructField('sender_id', StringType(), True),
    StructField('receiver_id', StringType(), True),
    StructField('amount_fcfa', DoubleType(), True),
    StructField('transaction_type', StringType(), True),
    StructField('location', StringType(), True),
    StructField('is_flagged', BooleanType(), True),
])

In [4]:
raw_stream = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', 'kafka:29092') \
    .option('subscribe', 'transactions') \
    .option('startingOffsets', 'latest') \
    .load()

In [5]:
parsed_df = raw_stream \
    .selectExpr('CAST(value AS STRING) as json_str', 'timestamp as kafka_ts') \
    .select(from_json(col('json_str'), schema).alias('data'), col('kafka_ts')) \
    .select('data.*', 'kafka_ts')

df_with_watermark = parsed_df.withWatermark('timestamp', '2 minutes')

In [6]:
velocity_fraud = df_with_watermark \
    .groupBy(
        window(col('timestamp'), '5 minutes', '1 minute'),
        col('sender_id')
    ) \
    .agg(count('*').alias('tx_count')) \
    .filter(col('tx_count') > 5) \
    .select(
        col('sender_id'),
        col('window.start').alias('window_start'),
        col('window.end').alias('window_end'),
        col('tx_count'),
        lit('VELOCITY_FRAUD').alias('fraud_type'),
        current_timestamp().alias('detected_at')
    )

In [7]:
volume_fraud = df_with_watermark \
    .groupBy(
        window(col('timestamp'), '10 minutes', '2 minutes'),  # fenêtre glissante
        col('sender_id')
    ) \
    .agg(spark_sum('amount_fcfa').alias('total_amount')) \
    .filter(col('total_amount') > 500000) \
    .select(
        col('sender_id'),
        col('window.start').alias('window_start'),
        col('window.end').alias('window_end'),
        col('total_amount'),
        lit('VOLUME_FRAUD').alias('fraud_type'),
        current_timestamp().alias('detected_at')
    )

In [8]:
# ─── SINK 1 : Écriture vers le topic Kafka 'fraud-alerts' ─────
def write_to_kafka(df, fraud_type_label):
    return df \
        .select(to_json(struct('*')).alias('value')) \
        .writeStream \
        .format('kafka') \
        .option('kafka.bootstrap.servers', 'kafka:29092') \
        .option('topic', 'fraud-alerts') \
        .option('checkpointLocation', f'/tmp/checkpoint_{fraud_type_label}') \
        .outputMode('update') \
        .start()

# ─── SINK 2 : Écriture dans le Data Lake local (Parquet) ──────
def write_to_datalake(df, fraud_type_label):
    return df \
        .writeStream \
        .format('parquet') \
        .option('path', f'/tmp/waveguard_lake/{fraud_type_label}') \
        .option('checkpointLocation', f'/tmp/checkpoint_lake_{fraud_type_label}') \
        .outputMode('append') \
        .trigger(processingTime='30 seconds') \
        .start()

In [ ]:
q1_kafka = write_to_kafka(velocity_fraud, 'velocity')
q2_kafka = write_to_kafka(volume_fraud, 'volume')
q1_lake = write_to_datalake(velocity_fraud, 'velocity')
q2_lake = write_to_datalake(volume_fraud, 'volume')

# Sink console EN PLUS (pas dans le sujet), juste pour voir les résultats
# directement dans Jupyter pendant les tests
q_console_velocity = velocity_fraud.writeStream \
    .format('console') \
    .outputMode('update') \
    .option('truncate', 'false') \
    .start()

q_console_volume = volume_fraud.writeStream \
    .format('console') \
    .outputMode('update') \
    .option('truncate', 'false') \
    .start()

spark.streams.awaitAnyTermination()

In [17]:
print(spark.version)

3.5.0
